In [0]:
# Databricks notebook source
#  %md
#  # 09 · Gold: ranking de ativos com controle estatístico
#  **Fase 3 · Gold** | Entrada: `gold.dispositivo_hora`
# 
#  O diagnóstico 07b mostrou que nenhum sinal (picos, alarme, tensão ausente, corrente sentinela, temperatura residual, corrente e tensão médias) separa os dispositivos além do acaso (razão observado/esperado entre 0,92 e 1,11). `alarme` é idêntico a `temp_pico`.
# 
#  Por isso o ranking **não** usa score ponderado com `PERCENT_RANK` (que ordenaria ruído). Cada ativo recebe um **z-score por sinal** contra a taxa da frota, e só é marcado como "atenção" se ultrapassar um limite corrigido para múltiplos testes (Bonferroni). O pipeline continua útil: se houver um ativo realmente anômalo (ou dados reais), ele aparece.
# 
#  - z = (taxa do ativo − taxa da frota) / erro padrão por acaso
#  - Sinais: sobretemperatura (`temp_pico` = alarme), tensão ausente, corrente sentinela
#  - Posição: `z_max` decrescente, desempate por `device_id`

from statistics import NormalDist

CATALOG = "project_lakehouse_telemetria_ele"
GOLD = f"{CATALOG}.gold"

# ---- PARÂMETROS ----
ALFA = 0.05                 # taxa de falso positivo global desejada
N_SINAIS = 3
N_ATIVOS = 50
# ---------------------
N_TESTES = N_SINAIS * N_ATIVOS
Z_LIMITE = round(NormalDist().inv_cdf(1 - ALFA / N_TESTES), 3)   # unilateral, Bonferroni
print(f"Z_LIMITE = {Z_LIMITE} ({N_TESTES} testes, alfa global {ALFA})")

In [0]:
#  1. Construir `gold.ranking_risco_ativos` (snapshot)


spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.ranking_risco_ativos AS
WITH t AS (
  SELECT device_id,
         SUM(leituras) AS leituras,
         SUM(qtd_picos) AS qtd_picos,
         SUM(qtd_tensao_ausente) AS qtd_tensao_ausente,
         SUM(qtd_corrente_sentinela) AS qtd_corrente_sentinela
  FROM {GOLD}.dispositivo_hora GROUP BY device_id
),
f AS (
  SELECT SUM(qtd_picos)/SUM(leituras) AS p_picos,
         SUM(qtd_tensao_ausente)/SUM(leituras) AS p_tensao,
         SUM(qtd_corrente_sentinela)/SUM(leituras) AS p_sent
  FROM t
),
z AS (
  SELECT t.*, f.p_picos, f.p_tensao, f.p_sent,
    (qtd_picos/leituras - p_picos)/SQRT(p_picos*(1-p_picos)/leituras) AS z_picos,
    (qtd_tensao_ausente/leituras - p_tensao)/SQRT(p_tensao*(1-p_tensao)/leituras) AS z_tensao,
    (qtd_corrente_sentinela/leituras - p_sent)/SQRT(p_sent*(1-p_sent)/leituras) AS z_sentinela
  FROM t CROSS JOIN f
),
m AS (
  SELECT *, GREATEST(z_picos, z_tensao, z_sentinela) AS z_max FROM z
),
p AS (
  SELECT *,
         ROW_NUMBER() OVER (ORDER BY z_max DESC, device_id) AS posicao,
         ROW_NUMBER() OVER (ORDER BY qtd_picos DESC, device_id) AS posicao_so_picos
  FROM m
)
SELECT device_id, posicao,
       ROUND(z_max,3) AS z_max,
       CASE WHEN z_max >= {Z_LIMITE} THEN 'atenção' ELSE 'dentro do esperado' END AS nivel,
       CASE WHEN z_max = z_picos THEN 'sobretemperatura'
            WHEN z_max = z_tensao THEN 'tensão ausente'
            ELSE 'corrente sentinela' END AS sinal_dominante,
       ROUND(z_picos,3) AS z_picos, ROUND(z_tensao,3) AS z_tensao, ROUND(z_sentinela,3) AS z_sentinela,
       posicao_so_picos, posicao_so_picos - posicao AS delta_posicao,
       leituras, qtd_picos, qtd_tensao_ausente, qtd_corrente_sentinela,
       ROUND(100*qtd_picos/leituras,3) AS pct_picos,
       ROUND(100*(qtd_picos/leituras - 1.96*SQRT(p_picos*(1-p_picos)/leituras)),3) AS pct_picos_ic95_min,
       ROUND(100*(qtd_picos/leituras + 1.96*SQRT(p_picos*(1-p_picos)/leituras)),3) AS pct_picos_ic95_max,
       ROUND(100*qtd_tensao_ausente/leituras,3) AS pct_tensao_ausente,
       ROUND(100*qtd_corrente_sentinela/leituras,3) AS pct_corrente_sentinela
FROM p
""")
print(spark.table(f"{GOLD}.ranking_risco_ativos").count(), "ativos no ranking")

In [0]:
# 2. Top 10 por z_max
#  `pct_picos_ic95_*` é o intervalo de confiança de 95% da taxa de picos do próprio ativo (sem correção para múltiplos testes). Se ele contém a taxa da frota (≈0,30%), o ativo não se distingue da frota naquele sinal. Com 50 ativos, é esperado que cerca de 2 a 3 fiquem fora por acaso, por isso o critério de decisão é o `z_limite` corrigido.

display(spark.sql(f"""
SELECT posicao, device_id, nivel, z_max, sinal_dominante, z_picos, z_tensao, z_sentinela,
       pct_picos, pct_picos_ic95_min, pct_picos_ic95_max
FROM {GOLD}.ranking_risco_ativos ORDER BY posicao LIMIT 10
"""))

In [0]:
# 3. Os z-scores se comportam como ruído?
#  Sob ruído puro, a média dos z fica perto de 0 e o desvio perto de 1. Desvio claramente acima de 1 indica diferença real entre ativos.

display(spark.sql(f"""
SELECT 'sobretemperatura' AS sinal, ROUND(AVG(z_picos),3) AS media_z, ROUND(STDDEV(z_picos),3) AS desvio_z, ROUND(MAX(z_picos),3) AS z_maximo FROM {GOLD}.ranking_risco_ativos
UNION ALL SELECT 'tensão ausente', ROUND(AVG(z_tensao),3), ROUND(STDDEV(z_tensao),3), ROUND(MAX(z_tensao),3) FROM {GOLD}.ranking_risco_ativos
UNION ALL SELECT 'corrente sentinela', ROUND(AVG(z_sentinela),3), ROUND(STDDEV(z_sentinela),3), ROUND(MAX(z_sentinela),3) FROM {GOLD}.ranking_risco_ativos
"""))

In [0]:
# 4. Quantos ativos passam do limite

display(spark.sql(f"""
SELECT nivel, COUNT(*) AS ativos, ROUND(MAX(z_max),3) AS z_max_do_grupo, {Z_LIMITE} AS z_limite
FROM {GOLD}.ranking_risco_ativos GROUP BY nivel
"""))

# 5. Leitura
#  Se todos os ativos ficarem em "dentro do esperado", o resultado é: a diferença entre o TRF-027 (80 picos) e a média (63) é compatível com sorteio aleatório de 50 ativos. O ranking existe como ferramenta de monitoramento, mas neste dataset sintético não há ativo que mereça atenção por estatística.